# BF6 — Dwell Time Prediction + Next Artwork Prediction

Two prediction tasks using semantic signals from the B0-B6 pipeline.

**Part A — Dwell Time Prediction:**
- k-NN on 28-dim visitor features and 41-dim joint space
- Baselines: global mean, per-artwork mean, per-visitor mean
- Metrics: MAE, RMSE, Spearman correlation

**Part B — Next Artwork Prediction:**
- 4-layer predictor: base flow → context blend → covisit smoothing → category fallback
- Floor penalty 0.7x for different-floor candidates
- Metrics: Hit@1/5/10, MRR

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), 'helpers'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict

from helpers.museum_recommender import MuseumRecommender

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

In [ ]:
rec = MuseumRecommender()
rec.load_all_data(features_path='output/semantic_features/trajectory_features.json')

print(f"Trajectories: {len(rec.sequences)}")
print(f"Artworks: {len(rec.all_artwork_uris)}")
print(f"Interaction records: {sum(len(v) for v in rec.interaction_data.values())}")

---
# Part A — Dwell Time Prediction

## A1. Dwell Distribution Analysis

In [ ]:
# Collect all dwell times with metadata
dwell_records = []
for traj, interactions in rec.interaction_data.items():
    meta = rec.traj_metadata.get(traj, {})
    nav_styles = meta.get('nav_styles', set())
    nav_label = list(nav_styles)[0] if nav_styles else 'NoStyle'
    engagement = meta.get('engagement', 'Unknown')

    for item in interactions:
        if item['dwell'] > 0 and item['int_type'] in (
            'MeaningfulInteraction', 'ImmersiveInteraction', 'DominantInteraction'
        ):
            dwell_records.append({
                'traj': traj,
                'poi': item['poi'],
                'dwell': item['dwell'],
                'category': item['category'],
                'floor': item['floor'],
                'int_type': item['int_type'],
                'nav_style': nav_label,
                'engagement': engagement,
            })

dwell_df = pd.DataFrame(dwell_records)
print(f"Dwell records: {len(dwell_df)}")
print("\nDwell statistics (seconds):")
print(dwell_df['dwell'].describe())

In [ ]:
# Dwell by category
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

cat_dwell = dwell_df.groupby('category')['dwell'].agg(['mean', 'median', 'count'])
cat_dwell = cat_dwell.sort_values('count', ascending=False)
cat_dwell[['mean', 'median']].plot(kind='bar', ax=axes[0])
axes[0].set_title('Dwell Time by Category')
axes[0].set_ylabel('Seconds')
axes[0].tick_params(axis='x', rotation=45)

# Dwell by navigation style
nav_dwell = dwell_df.groupby('nav_style')['dwell'].agg(['mean', 'median', 'count'])
nav_dwell[['mean', 'median']].plot(kind='bar', ax=axes[1])
axes[1].set_title('Dwell Time by Navigation Style')
axes[1].set_ylabel('Seconds')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Dwell by floor
floor_dwell = dwell_df[dwell_df['floor'] >= 0].groupby('floor')['dwell'].agg(['mean', 'median', 'std', 'count'])
print("Dwell by floor:")
display(floor_dwell)

# Significance correlation
sig_arts = rec._get_significant_artworks()
dwell_df['is_significant'] = dwell_df['poi'].isin(sig_arts)
sig_comparison = dwell_df.groupby('is_significant')['dwell'].agg(['mean', 'median', 'count'])
print("\nDwell by significance:")
display(sig_comparison)

## A2. Feature Engineering

Visitor features (28-dim) + artwork features (category 8-dim, floor 3-dim, significance 1-dim, popularity 1-dim) = 41-dim joint space.

In [ ]:
# Show feature composition
print("Visitor features (28 dims):")
print("  Artwork type distribution: 8")
print("  Floor distribution: 3")
print("  Navigation style: 4")
print("  Engagement metrics: 8")
print("  Interaction types: 5")
print("\nArtwork features (13 dims):")
print("  Category one-hot: 8")
print("  Floor one-hot: 3")
print("  Significance: 1")
print("  Popularity: 1")
print("\nJoint space: 41 dims")

# Example artwork feature vector
sample_art = list(rec.all_artwork_uris)[0]
art_vec = rec._build_artwork_feature_vector(sample_art)
art_meta = rec.poi_metadata.get(sample_art, {})
print(f"\nExample: {art_meta.get('label', '?')} ({art_meta.get('category', '?')}, floor {art_meta.get('floor', '?')})")
print(f"  Artwork vector: {art_vec}")

## A3. Three Prediction Methods

In [ ]:
# Example predictions for one visitor
sample_traj = list(rec.sequences.keys())[0]
sample_interactions = rec.interaction_data[sample_traj]
sample_meaningful = [i for i in sample_interactions
                     if i['int_type'] in ('MeaningfulInteraction', 'ImmersiveInteraction') and i['dwell'] > 0]

if len(sample_meaningful) >= 3:
    test_art = sample_meaningful[-1]['poi']
    actual_dwell = sample_meaningful[-1]['dwell']
    art_label = rec.poi_metadata.get(test_art, {}).get('label', '?')

    print(f"Predicting dwell for: {art_label}")
    print(f"Actual dwell: {actual_dwell:.1f}s\n")

    # Method 1: Visitor-based k-NN
    pred1 = rec.predict_dwell(sample_traj, test_art, k=5)
    print(f"Visitor-based k-NN (k=5): {pred1['predicted_dwell']}s ({pred1['method']})")
    for n in pred1['neighbors']:
        print(f"  Neighbor: sim={n['sim']}, dwell={n['dwell']}s")

    # Method 2: Combined k-NN (41-dim)
    pred2 = rec.predict_dwell_combined(sample_traj, test_art, k=5)
    print(f"\nCombined k-NN (k=5): {pred2['predicted_dwell']}s ({pred2['method']})")

    # Method 3: Category-stratified baseline
    cat = rec.poi_metadata.get(test_art, {}).get('category', '')
    cat_dwells = dwell_df[dwell_df['category'] == cat]['dwell']
    print(f"\nCategory baseline ({cat}): mean={cat_dwells.mean():.1f}s, median={cat_dwells.median():.1f}s")

## A4. Evaluation — Two Protocols

In [ ]:
# Trajectory split evaluation
print("=== Dwell Prediction: Trajectory Split (70/30) ===")
dwell_ts = rec.evaluate_dwell_prediction(
    k_range=[3, 5, 7, 10, 15], protocol='trajectory_split'
)

# Display results table
dwell_rows = []
for key, metrics in sorted(dwell_ts.items()):
    dwell_rows.append({
        'Method': key,
        'MAE (s)': metrics['mae'],
        'RMSE (s)': metrics['rmse'],
        'Spearman': metrics['spearman'],
        'N': metrics['n'],
    })
display(pd.DataFrame(dwell_rows))

In [ ]:
# Leave-k-out evaluation
print("\n=== Dwell Prediction: Leave-5-out ===")
dwell_lko = rec.evaluate_dwell_prediction(
    k_range=[3, 5, 7, 10, 15], protocol='leave_k_out', leave_k=5
)

dwell_rows_lko = []
for key, metrics in sorted(dwell_lko.items()):
    dwell_rows_lko.append({
        'Method': key,
        'MAE (s)': metrics['mae'],
        'RMSE (s)': metrics['rmse'],
        'Spearman': metrics['spearman'],
        'N': metrics['n'],
    })
display(pd.DataFrame(dwell_rows_lko))

In [ ]:
# k sensitivity plot
k_values = [3, 5, 7, 10, 15]
mae_values = [dwell_ts.get(f'knn_k{k}', {}).get('mae', 0) for k in k_values]
spearman_values = [dwell_ts.get(f'knn_k{k}', {}).get('spearman', 0) for k in k_values]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(k_values, mae_values, 'bo-', linewidth=2)
# Per-artwork baseline
pa_mae = dwell_ts.get('per_artwork', {}).get('mae', 0)
ax1.axhline(y=pa_mae, color='red', linestyle='--', label=f'Per-artwork mean ({pa_mae:.1f})')
gm_mae = dwell_ts.get('global_mean', {}).get('mae', 0)
ax1.axhline(y=gm_mae, color='orange', linestyle='--', label=f'Global mean ({gm_mae:.1f})')
ax1.set_xlabel('k')
ax1.set_ylabel('MAE (seconds)')
ax1.set_title('Dwell MAE vs k')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(k_values, spearman_values, 'go-', linewidth=2)
ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('k')
ax2.set_ylabel('Spearman Correlation')
ax2.set_title('Rank Correlation vs k')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Per-category MAE (trajectory split, k=5)
cat_errors = defaultdict(list)
eligible = [t for t in rec.interaction_data if len(rec.interaction_data[t]) >= 5]

for traj in eligible:
    interactions = rec.interaction_data[traj]
    meaningful = [i for i in interactions
                  if i['int_type'] in ('MeaningfulInteraction', 'ImmersiveInteraction',
                                       'DominantInteraction') and i['dwell'] > 0]
    if len(meaningful) < 5:
        continue
    split_idx = max(2, int(len(meaningful) * 0.7))
    holdout = meaningful[split_idx:]
    for item in holdout:
        pred = rec.predict_dwell(traj, item['poi'], k=5)
        error = abs(item['dwell'] - pred['predicted_dwell'])
        cat_errors[item['category']].append(error)

print("Per-category MAE (k=5):")
for cat, errors in sorted(cat_errors.items(), key=lambda x: -len(x[1])):
    print(f"  {cat}: MAE={np.mean(errors):.1f}s (n={len(errors)})")

## A5. Prediction Examples

In [ ]:
# 3 visitors of different styles — predict dwell for 5 unvisited artworks
sample_trajs = []
for traj, meta in rec.traj_metadata.items():
    styles = meta.get('nav_styles', set())
    if 'ExploratoryMovement' in styles and len(sample_trajs) < 1:
        sample_trajs.append(traj)
    elif 'ImmersiveMovement' in styles and len(sample_trajs) < 2:
        sample_trajs.append(traj)
    elif 'CasualMovement' in styles and len(sample_trajs) < 3:
        sample_trajs.append(traj)

all_t = list(rec.sequences.keys())
while len(sample_trajs) < 3 and all_t:
    t = all_t.pop(0)
    if t not in sample_trajs:
        sample_trajs.append(t)

for traj in sample_trajs:
    meta = rec.traj_metadata.get(traj, {})
    visited = rec.traj_artwork_sets.get(traj, set())
    unvisited = [a for a in rec.all_artwork_uris if a not in visited]
    np.random.seed(42)
    sample_arts = np.random.choice(unvisited, size=min(5, len(unvisited)), replace=False)

    print(f"\n{'='*60}")
    print(f"Visitor: {traj.split('/')[-1]} ({meta.get('nav_styles', set())})")
    for art in sample_arts:
        pred = rec.predict_dwell(traj, art, k=5)
        art_meta = rec.poi_metadata.get(art, {})
        label = art_meta.get('label', '?')[:40]
        cat = art_meta.get('category', '?')
        print(f"  {label} ({cat}): predicted {pred['predicted_dwell']}s")
        if pred['neighbors']:
            dwells_str = ', '.join(f"{n['dwell']}s(sim={n['sim']})" for n in pred['neighbors'][:3])
            print(f"    Based on: {dwells_str}")

---
# Part B — Next Artwork Prediction

## B1. Transition Analysis

In [ ]:
# Top-20 artwork transitions
transition_counts = Counter()
for traj, seq in rec.sequences.items():
    for i in range(len(seq) - 1):
        src_label = rec.poi_metadata.get(seq[i], {}).get('label', '?')[:25]
        dst_label = rec.poi_metadata.get(seq[i+1], {}).get('label', '?')[:25]
        transition_counts[(src_label, dst_label)] += 1

print("Top 20 artwork transitions:")
for (src, dst), count in transition_counts.most_common(20):
    print(f"  {src} -> {dst}: {count}")

In [ ]:
# Category transition matrix (8x8)
cat_transitions = defaultdict(Counter)
for traj, seq in rec.sequences.items():
    for i in range(len(seq) - 1):
        src_cat = rec.poi_metadata.get(seq[i], {}).get('category', 'Unknown')
        dst_cat = rec.poi_metadata.get(seq[i+1], {}).get('category', 'Unknown')
        cat_transitions[src_cat][dst_cat] += 1

cats = sorted(set(c for trans in cat_transitions.values() for c in trans) | set(cat_transitions.keys()))
trans_matrix = pd.DataFrame(0, index=cats, columns=cats)
for src, targets in cat_transitions.items():
    total = sum(targets.values())
    for dst, count in targets.items():
        trans_matrix.loc[src, dst] = count / total

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(trans_matrix, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax)
ax.set_title('Category Transition Probabilities')
ax.set_ylabel('From')
ax.set_xlabel('To')
plt.tight_layout()
plt.show()

In [ ]:
# Floor transition patterns
floor_transitions = defaultdict(Counter)
for traj, seq in rec.sequences.items():
    for i in range(len(seq) - 1):
        src_floor = rec.poi_metadata.get(seq[i], {}).get('floor', -1)
        dst_floor = rec.poi_metadata.get(seq[i+1], {}).get('floor', -1)
        if src_floor >= 0 and dst_floor >= 0:
            floor_transitions[src_floor][dst_floor] += 1

print("Floor transition counts:")
for src in sorted(floor_transitions):
    total = sum(floor_transitions[src].values())
    probs = {dst: f"{count/total:.2f}" for dst, count in sorted(floor_transitions[src].items())}
    same_floor_pct = floor_transitions[src].get(src, 0) / total * 100
    print(f"  Floor {src}: {probs} (same floor: {same_floor_pct:.0f}%)")

In [ ]:
# Markov stationarity check
# First-order vs second-order: compare P(next|current) vs P(next|current,previous)
first_order = defaultdict(Counter)
second_order = defaultdict(Counter)

for traj, seq in rec.sequences.items():
    for i in range(len(seq) - 1):
        first_order[seq[i]][seq[i+1]] += 1
    for i in range(len(seq) - 2):
        second_order[(seq[i], seq[i+1])][seq[i+2]] += 1

print(f"First-order states: {len(first_order)}")
print(f"Second-order states: {len(second_order)}")
print(f"Ratio: {len(second_order)/max(len(first_order),1):.1f}x")

# Average entropy (uncertainty) comparison
from scipy.stats import entropy as sp_entropy

fo_entropies = []
for state, targets in first_order.items():
    total = sum(targets.values())
    probs = [c/total for c in targets.values()]
    fo_entropies.append(sp_entropy(probs))

so_entropies = []
for state, targets in second_order.items():
    total = sum(targets.values())
    probs = [c/total for c in targets.values()]
    so_entropies.append(sp_entropy(probs))

print(f"\nAvg entropy — 1st order: {np.mean(fo_entropies):.3f}, 2nd order: {np.mean(so_entropies):.3f}")
print(f"Entropy reduction: {(1 - np.mean(so_entropies)/np.mean(fo_entropies))*100:.1f}%")

## B2. 4-Layer Predictor

In [ ]:
# Example prediction for a trajectory
sample_traj = list(rec.sequences.keys())[0]
seq = rec.sequences[sample_traj]
meta = rec.traj_metadata.get(sample_traj, {})

# Use first 5 artworks as input
input_seq = seq[:5]
context = {}
nav_styles = meta.get('nav_styles', set())
if nav_styles:
    context['nav_style'] = list(nav_styles)[0]

preds = rec.predict_next_artwork(input_seq, context=context, top_k=10)

print(f"Input sequence ({len(input_seq)} artworks):")
for i, poi in enumerate(input_seq, 1):
    label = rec.poi_metadata.get(poi, {}).get('label', '?')[:40]
    floor = rec.poi_metadata.get(poi, {}).get('floor', '?')
    print(f"  {i}. {label} (floor {floor})")

actual_next = seq[5] if len(seq) > 5 else None
if actual_next:
    actual_label = rec.poi_metadata.get(actual_next, {}).get('label', '?')[:40]
    print(f"\nActual next: {actual_label}")

print("\nPredictions (top 10):")
for i, p in enumerate(preds, 1):
    hit = ' <<<' if p['poi'] == actual_next else ''
    print(f"  {i}. {p['label'][:40]} ({p['category']}, floor {p['floor']}) "
          f"prob={p['probability']:.3f} [{p['source']}]{hit}")

## B3. Category Sequence Patterns

In [ ]:
# 2-gram and 3-gram mining
bigrams = Counter()
trigrams = Counter()

for traj, seq in rec.sequences.items():
    cat_seq = [rec.poi_metadata.get(poi, {}).get('category', 'Unknown') for poi in seq]
    for i in range(len(cat_seq) - 1):
        bigrams[(cat_seq[i], cat_seq[i+1])] += 1
    for i in range(len(cat_seq) - 2):
        trigrams[(cat_seq[i], cat_seq[i+1], cat_seq[i+2])] += 1

total_bigrams = sum(bigrams.values())
total_trigrams = sum(trigrams.values())

print("Top 15 category bigrams:")
for gram, count in bigrams.most_common(15):
    support = count / total_bigrams
    print(f"  {' -> '.join(gram)}: {count} (support={support:.3f})")

print("\nTop 10 category trigrams (support > 0.01):")
for gram, count in trigrams.most_common(10):
    support = count / total_trigrams
    if support > 0.01:
        print(f"  {' -> '.join(gram)}: {count} (support={support:.3f})")

## B4. Evaluation — Three Protocols

In [ ]:
# Trajectory split evaluation
print("=== Next Artwork: Trajectory Split (70/30) ===")
next_ts = rec.evaluate_next_prediction(
    protocol='trajectory_split', k_values=[1, 5, 10]
)
for k in [1, 5, 10]:
    ci = next_ts.get(f'hit@{k}_ci', (0, 0))
    print(f"Hit@{k}: {next_ts.get(f'hit@{k}', 0):.4f} [{ci[0]:.4f}, {ci[1]:.4f}]")
print(f"MRR: {next_ts.get('mrr', 0):.4f} {next_ts.get('mrr_ci', (0,0))}")
print(f"Steps evaluated: {next_ts.get('n_steps', 0)}")

print("\nPer navigation style:")
for nav, metrics in next_ts.get('nav_breakdown', {}).items():
    h5 = metrics.get('hit@5', 0)
    print(f"  {nav} (n={metrics.get('n', 0)}): Hit@5={h5:.4f}")

In [ ]:
# Leave-k-out evaluation
print("\n=== Next Artwork: Leave-5-out ===")
next_lko = rec.evaluate_next_prediction(
    protocol='leave_k_out', leave_k=5, k_values=[1, 5, 10]
)
for k in [1, 5, 10]:
    ci = next_lko.get(f'hit@{k}_ci', (0, 0))
    print(f"Hit@{k}: {next_lko.get(f'hit@{k}', 0):.4f} [{ci[0]:.4f}, {ci[1]:.4f}]")
print(f"MRR: {next_lko.get('mrr', 0):.4f}")

In [ ]:
# Baselines for comparison
print("\n=== Baseline Comparison (Trajectory Split) ===")

# Popularity baseline
pop_hits = {k: [] for k in [1, 5, 10]}
pop_mrr = []
eligible = [t for t in rec.sequences if len(rec.sequences[t]) >= 5]

art_popularity = Counter()
for arts in rec.traj_artwork_sets.values():
    for a in arts:
        art_popularity[a] += 1

for traj in eligible:
    seq = rec.sequences[traj]
    split_idx = max(2, int(len(seq) * 0.7))
    input_seq = seq[:split_idx]
    holdout_seq = seq[split_idx:]
    if not holdout_seq:
        continue

    current = list(input_seq)
    for actual_next in holdout_seq:
        visited_set = set(current)
        pop_ranked = [poi for poi, _ in art_popularity.most_common() if poi not in visited_set]

        for k in [1, 5, 10]:
            pop_hits[k].append(1.0 if actual_next in pop_ranked[:k] else 0.0)

        rr = 0.0
        for rank, poi in enumerate(pop_ranked, 1):
            if poi == actual_next:
                rr = 1.0 / rank
                break
        pop_mrr.append(rr)
        current.append(actual_next)

print("Popularity baseline:")
for k in [1, 5, 10]:
    print(f"  Hit@{k}: {np.mean(pop_hits[k]):.4f}")
print(f"  MRR: {np.mean(pop_mrr):.4f}")

print("\n4-layer predictor:")
for k in [1, 5, 10]:
    print(f"  Hit@{k}: {next_ts.get(f'hit@{k}', 0):.4f}")
print(f"  MRR: {next_ts.get('mrr', 0):.4f}")

## B5. Learning Curve — Accuracy vs Position

In [ ]:
learning_curve = rec.evaluate_next_learning_curve(positions=[3, 5, 10, 20, 30])

positions = sorted(learning_curve.keys())
hit1_vals = [learning_curve[p]['hit@1'] for p in positions]
hit5_vals = [learning_curve[p]['hit@5'] for p in positions]
mrr_vals = [learning_curve[p]['mrr'] for p in positions]
n_vals = [learning_curve[p]['n'] for p in positions]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(positions, hit1_vals, 'ro-', label='Hit@1', linewidth=2)
ax1.plot(positions, hit5_vals, 'bs-', label='Hit@5', linewidth=2)
ax1.plot(positions, mrr_vals, 'g^-', label='MRR', linewidth=2)
ax1.set_xlabel('Artworks Observed')
ax1.set_ylabel('Score')
ax1.set_title('Prediction Quality vs Visit Progress')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.bar(positions, n_vals, color='steelblue', alpha=0.7)
ax2.set_xlabel('Artworks Observed')
ax2.set_ylabel('Trajectories with >= N artworks')
ax2.set_title('Sample Size at Each Position')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nLearning curve data:")
for p in positions:
    d = learning_curve[p]
    print(f"  After {p} artworks (n={d['n']}): Hit@1={d['hit@1']:.3f}, Hit@5={d['hit@5']:.3f}, MRR={d['mrr']:.3f}")

## B6. Prediction Visualization — Actual vs Predicted

In [ ]:
# 3 trajectories: actual vs predicted as parallel timelines
viz_trajs = list(rec.sequences.keys())[:3]

for traj in viz_trajs:
    seq = rec.sequences[traj]
    meta = rec.traj_metadata.get(traj, {})
    if len(seq) < 8:
        continue

    split_idx = max(2, int(len(seq) * 0.7))
    input_seq = seq[:split_idx]
    holdout = seq[split_idx:]

    context = {}
    nav_styles = meta.get('nav_styles', set())
    if nav_styles:
        context['nav_style'] = list(nav_styles)[0]

    print(f"\n{'='*70}")
    print(f"Trajectory: {traj.split('/')[-1]} (nav: {nav_styles})")
    print(f"Input: {len(input_seq)} artworks | Holdout: {len(holdout)} artworks")

    # Predict step by step
    current = list(input_seq)
    print(f"\n{'Step':<5} {'Actual':<40} {'Predicted Top-1':<40} {'Hit?':<5}")
    print('-' * 90)

    for step, actual in enumerate(holdout, 1):
        preds = rec.predict_next_artwork(current, context=context, top_k=5)
        pred_top1 = preds[0]['label'][:38] if preds else '(none)'
        pred_top1_uri = preds[0]['poi'] if preds else None
        actual_label = rec.poi_metadata.get(actual, {}).get('label', '?')[:38]
        hit = 'Y' if pred_top1_uri == actual else ('~' if actual in [p['poi'] for p in preds[:5]] else 'N')
        print(f"{step:<5} {actual_label:<40} {pred_top1:<40} {hit:<5}")
        current.append(actual)

## B7. Trajectory Maps — Actual vs Predicted on Museum Floor

Spatial visualization using indoor coordinates (easting/northing). Blue path = visited sequence, green diamonds = actual holdout artworks, red stars = predicted next artworks. All museum artworks shown as faint background dots for spatial context.

In [ ]:
# Museum overview — all artwork positions colored by floor
fig, ax = plt.subplots(figsize=(14, 10))

floor_colors = {0: '#1976D2', 1: '#FF9800', 2: '#4CAF50'}
floor_labels = {0: 'Floor 0', 1: 'Floor 1', 2: 'Floor 2'}

for poi, c in rec.artwork_coords.items():
    color = floor_colors.get(c['floor'], 'gray')
    ax.scatter(c['x'], c['y'], c=color, s=12, alpha=0.4, zorder=1)

for fl, color in floor_colors.items():
    count = sum(1 for c in rec.artwork_coords.values() if c['floor'] == fl)
    ax.scatter([], [], c=color, s=30, alpha=0.6, label=f'{floor_labels[fl]} ({count} artworks)')

ax.set_title(f'Museum Floor Plan — {len(rec.artwork_coords)} Artwork Positions')
ax.set_xlabel('Easting (px)')
ax.set_ylabel('Northing (px)')
ax.legend(fontsize=9)
ax.set_aspect('equal')
ax.grid(True, alpha=0.15)
plt.tight_layout()
plt.show()

print(f"Coordinate ranges: x=[{min(c['x'] for c in rec.artwork_coords.values()):.0f}, "
      f"{max(c['x'] for c in rec.artwork_coords.values()):.0f}], "
      f"y=[{min(c['y'] for c in rec.artwork_coords.values()):.0f}, "
      f"{max(c['y'] for c in rec.artwork_coords.values()):.0f}]")

In [ ]:
# Actual vs predicted maps for 3 diverse trajectories
# Pick trajectories with different navigation styles and sufficient length
map_trajs = []
for traj, meta in rec.traj_metadata.items():
    if len(rec.sequences.get(traj, [])) < 8:
        continue
    if traj not in rec.position_coords or not rec.position_coords[traj]:
        continue
    styles = meta.get('nav_styles', set())
    if 'ExploratoryMovement' in styles and len(map_trajs) < 1:
        map_trajs.append(traj)
    elif 'ImmersiveMovement' in styles and len(map_trajs) < 2:
        map_trajs.append(traj)
    elif not styles and len(map_trajs) < 3:
        map_trajs.append(traj)

# Fill remaining
for traj in rec.sequences:
    if len(map_trajs) >= 3:
        break
    if traj not in map_trajs and len(rec.sequences[traj]) >= 8 and traj in rec.position_coords:
        map_trajs.append(traj)

for traj in map_trajs:
    fig = rec.plot_actual_vs_predicted_map(traj, split_ratio=0.7, top_k=5)
    plt.show()
    print()

In [ ]:
# Per-floor detail view for one trajectory
# Shows each floor separately to avoid overlapping coordinates
detail_traj = map_trajs[0] if map_trajs else list(rec.sequences.keys())[0]

# Determine which floors this trajectory visits
visited_floors = set()
for coord in rec.position_coords.get(detail_traj, []):
    visited_floors.add(coord['floor'])
visited_floors = sorted(visited_floors)

if len(visited_floors) > 1:
    fig, axes = plt.subplots(1, len(visited_floors), figsize=(7 * len(visited_floors), 8))
    if len(visited_floors) == 1:
        axes = [axes]

    input_seq, holdout = rec.split_trajectory(detail_traj, 0.7)
    meta = rec.traj_metadata.get(detail_traj, {})
    context = {}
    nav_styles = meta.get('nav_styles', set())
    if nav_styles:
        context['nav_style'] = list(nav_styles)[0]

    preds = rec.predict_next_artwork(input_seq, context=context, top_k=5)
    pred_pois = [p['poi'] for p in preds]

    for i, floor in enumerate(visited_floors):
        rec.plot_trajectory_on_floor(
            detail_traj,
            predicted_pois=pred_pois,
            holdout_pois=holdout,
            floor_filter=floor,
            ax=axes[i],
        )

    plt.suptitle(f'Per-Floor View: {detail_traj.split("/")[-1]}', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print(f"Trajectory {detail_traj.split('/')[-1]} only visits floor(s): {visited_floors}")